# 04 — Hybrid Integration & Sparsity (UBCF)

Protocol D (user-based):
- Active users: classical UBCF
- Sparse/new users: quantum-cluster-guided neighbors
- Extreme cold users: content-profile fallback (genres/tags)
- Sparsity simulations: 20% to 95% history drop
- Long-tail metrics: coverage, gini, diversity

Output: `hybrid_ubcf_recs.csv`

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

IN_DIR = Path("data/processed_32m_ubcf")
OUT_DIR = Path("data/processed_32m_ubcf")

SEED = 42
TOPN = 10
EVAL_USERS = 300
SPARSE_NNZ = 20
EXTREME_COLD_NNZ = 5
SPARSITY_DROPS = [0.20, 0.40, 0.60, 0.80, 0.90, 0.95]

user_sparse = load_npz(IN_DIR / "user_sparse.npz").tocsr().astype(np.float32)
user_latent = np.load(IN_DIR / "user_latent.npy").astype(np.float32)
content_features = np.load(IN_DIR / "content_features.npy").astype(np.float32)
classical_clusters = np.load(IN_DIR / "user_clusters.npy").astype(np.int32)
quantum_labels = np.load(IN_DIR / "quantum_user_labels.npy").astype(np.int32)
kernel_data = np.load(IN_DIR / "user_kernel_matrix.npz")
sample_idx = kernel_data["sample_idx"].astype(np.int32)

with open(IN_DIR / "id_maps.pkl", "rb") as f:
    idmap = pickle.load(f)

unique_users = idmap["unique_users"]
unique_movies = idmap["unique_movies"]

print(user_sparse.shape, content_features.shape)

In [ ]:
# Build quantum-cluster assignment for all users from latent centroids
sample_latent = user_latent[sample_idx]
q_ids = np.unique(quantum_labels)
q_centroids = np.vstack([sample_latent[quantum_labels == q].mean(axis=0) for q in q_ids]).astype(np.float32)
dist = ((user_latent[:, None, :] - q_centroids[None, :, :]) ** 2).sum(axis=2)
user_q_assign = np.argmin(dist, axis=1).astype(np.int32)

user_nnz = np.diff(user_sparse.indptr)
rng = np.random.default_rng(SEED)
eligible = np.where(user_nnz >= 3)[0]
eval_users = rng.choice(eligible, size=min(EVAL_USERS, len(eligible)), replace=False)

movie_pop = np.asarray((user_sparse != 0).sum(axis=0)).ravel()
top_candidates = np.argsort(movie_pop)[::-1][:3000]

def user_mean(u):
    row = user_sparse[u]
    return float(row.data.mean()) if row.nnz else 3.5

def pred_classical_active(u, m, neighbors):
    um = user_mean(u)
    num, den = 0.0, 0.0
    for v, s in neighbors:
        row = user_sparse[v]
        vm = float(row.data.mean()) if row.nnz else 3.5
        rating_map = {i:r for i, r in zip(row.indices, row.data)}
        if m in rating_map:
            num += s * (float(rating_map[m]) - vm)
            den += abs(s)
    return float(np.clip(um + num / (den + 1e-8), 0.5, 5.0))

def pred_content_cold(u, m, seen_items):
    if len(seen_items) == 0:
        return 0.0
    idx = np.array(list(seen_items), dtype=np.int32)
    profile = content_features[idx].mean(axis=0)
    den = (np.linalg.norm(profile) * np.linalg.norm(content_features[m])) + 1e-8
    return float(np.dot(profile, content_features[m]) / den)

In [ ]:
# Hybrid recommendations + sparsity stress
rows = []
all_lists = []
stress_rows = []

for drop in [0.0] + SPARSITY_DROPS:
    rng_drop = np.random.default_rng(SEED + int(drop * 1000))
    hits, ndcgs, precs = [], [], []

    for u in eval_users:
        start, end = user_sparse.indptr[u], user_sparse.indptr[u+1]
        seen = user_sparse.indices[start:end]
        if len(seen) < 3:
            continue

        held = int(rng_drop.choice(seen))
        train_seen = [x for x in seen.tolist() if x != held]
        keep_n = max(1, int(np.ceil((1.0 - drop) * len(train_seen))))
        keep_idx = rng_drop.choice(np.arange(len(train_seen)), size=keep_n, replace=False)
        seen_set = set(np.array(train_seen)[keep_idx].tolist())

        # Protocol D routing by user profile density
        nnz = len(seen_set)
        if nnz >= SPARSE_NNZ:
            neigh = np.where(classical_clusters == classical_clusters[u])[0]
            strategy = "classical_active"
        elif nnz >= EXTREME_COLD_NNZ:
            neigh = np.where(user_q_assign == user_q_assign[u])[0]
            strategy = "quantum_sparse"
        else:
            neigh = np.array([], dtype=np.int32)
            strategy = "content_cold"

        neigh = neigh[neigh != u]
        sims = []
        if len(neigh) > 0:
            urow = user_sparse[u]
            umap = {i:r for i, r in zip(urow.indices, urow.data)}
            for v in neigh[:200]:
                vrow = user_sparse[v]
                common = np.intersect1d(urow.indices, vrow.indices, assume_unique=False)
                if len(common) < 2:
                    continue
                vmap = {i:r for i, r in zip(vrow.indices, vrow.data)}
                uv = np.array([umap[i] for i in common], dtype=np.float32)
                vv = np.array([vmap[i] for i in common], dtype=np.float32)
                uc = uv - uv.mean()
                vc = vv - vv.mean()
                s = float(np.dot(uc, vc) / (np.linalg.norm(uc) * np.linalg.norm(vc) + 1e-8))
                sims.append((int(v), s))
            sims.sort(key=lambda x: abs(x[1]), reverse=True)
            sims = sims[:40]

        cand = [m for m in top_candidates if m not in seen_set]
        scored = []
        for m in cand:
            if strategy == "content_cold":
                sc = pred_content_cold(u, int(m), seen_set)
            else:
                sc = pred_classical_active(u, int(m), sims)
            scored.append((int(m), float(sc), strategy))

        scored.sort(key=lambda x: x[1], reverse=True)
        top = scored[:TOPN]
        top_items = [t[0] for t in top]

        hit = 1.0 if held in top_items else 0.0
        hits.append(hit)
        precs.append(hit / TOPN)
        if hit:
            rank = top_items.index(held) + 1
            ndcgs.append(float(1.0 / np.log2(rank + 1)))
        else:
            ndcgs.append(0.0)

        if drop == 0.0:
            all_lists.append([{"item_idx": t[0], "score": t[1], "strategy": t[2]} for t in top])
            uid = int(unique_users[u]) if u < len(unique_users) else int(u)
            for r, (mi, sc, st) in enumerate(top, start=1):
                rows.append({"user_idx": int(u), "user_id": uid, "movie_id": int(unique_movies[mi]), "item_idx": int(mi), "rank": int(r), "score": float(sc), "strategy": st})

    stress_rows.append({"sparsity_drop": float(drop), "hr@10": float(np.mean(hits)) if hits else np.nan, "ndcg@10": float(np.mean(ndcgs)) if ndcgs else np.nan, "precision@10": float(np.mean(precs)) if precs else np.nan})

hybrid_recs_df = pd.DataFrame(rows)
stress_df = pd.DataFrame(stress_rows)

# Long-tail metrics on primary (drop=0) recs
counts = hybrid_recs_df["item_idx"].value_counts().reindex(np.arange(user_sparse.shape[1]), fill_value=0).values.astype(np.float64)
coverage = float(hybrid_recs_df["item_idx"].nunique() / user_sparse.shape[1]) if len(hybrid_recs_df) else 0.0
x = np.sort(counts)
gini = float((2*np.sum(np.arange(1,len(x)+1)*x)/(len(x)*np.sum(x)+1e-12))-((len(x)+1)/len(x))) if np.sum(x)>0 else 0.0

divs = []
for lst in all_lists:
    items = [d["item_idx"] for d in lst]
    if len(items) < 2:
        continue
    sim = cosine_similarity(content_features[np.array(items, dtype=np.int32)])
    iu = np.triu_indices_from(sim, k=1)
    if len(iu[0]):
        divs.append(float(1.0 - np.mean(sim[iu])))
diversity = float(np.mean(divs)) if divs else 0.0

stress_df["coverage"] = np.nan
stress_df["gini"] = np.nan
stress_df["diversity"] = np.nan
stress_df.loc[stress_df["sparsity_drop"] == 0.0, ["coverage", "gini", "diversity"]] = [coverage, gini, diversity]

hybrid_recs_df.to_csv(OUT_DIR / "hybrid_ubcf_recs.csv", index=False)
stress_df.to_csv(OUT_DIR / "hybrid_ubcf_stress.csv", index=False)

plt.figure(figsize=(8,4))
plt.plot(stress_df["sparsity_drop"]*100, stress_df["hr@10"], marker="o", label="HR@10")
plt.plot(stress_df["sparsity_drop"]*100, stress_df["ndcg@10"], marker="o", label="NDCG@10")
plt.plot(stress_df["sparsity_drop"]*100, stress_df["precision@10"], marker="o", label="Precision@10")
plt.title("UBCF Hybrid Protocol D under Sparsity")
plt.xlabel("Drop %")
plt.ylabel("Metric")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print(stress_df)
print("Saved: hybrid_ubcf_recs.csv")